In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 8.11 Real Band Structures: The Empirical Pseudopotential Method

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VIII — Electronic Structure and Many-Body Matter",
    number="8.11",
    title="Real Band Structures: The Empirical Pseudopotential Method",
    blurb="Six numbers and a matrix produce silicon. Cohen and Bergstresser's 1966 "
    "form factors — three per element, fitted to optical data — turn the "
    "plane-wave secular problem of 8.10 into the genuine band structures of Si "
    "and GaAs: the indirect gap with its conduction minimum famously parked at "
    "85% of the way to X, the direct gap that makes gallium arsenide glow, the "
    "ionicity splitting switched on and off by the antisymmetric factors, and "
    "the honest lesson that an empirical model's parameters belong to its "
    "basis. The volume's handshake with the companion MMM course happens here, "
    "at the silicon band structure both courses compute.",
    difficulty="advanced",
    estimate="130–160 min",
)

## Notebook overview

Everything Movement III has built converges on this notebook's promise: *real
materials*. The empirical pseudopotential method (EPM) of Cohen and
Bergstresser {cite}`cohenbergstresser1966` is the licensed shortcut of
[§8.10](plane-waves-pseudopotentials.ipynb): rather than constructing
potentials from atoms and fighting the nodal catch, fit the crystal's few
symmetry-allowed Fourier coefficients directly to measured optical gaps. For
diamond- and zinc-blende semiconductors only three symmetric (and, for the
ionic compounds, three antisymmetric) coefficients survive symmetry — six
numbers for GaAs, three for Si — and with them the full valence and low
conduction bands of fourteen semiconductors emerged in 1966 to
tenth-of-an-eV fidelity. It remains the fastest honest route from "lattice
constant plus a handful of parameters" to "the band structure in a textbook."

The build: the diamond/zinc-blende potential matrix (structure factors,
symmetric and antisymmetric channels) assembled and its symmetry content
verified at $\Gamma$; silicon's band structure along $L$–$\Gamma$–$X$ with
its triply degenerate valence top, $12.6$-eV valence width, and the
conduction minimum at $0.85$ of $\Gamma X$ — the indirect gap that makes
silicon silicon; the basis lesson (at the 1966 paper's own 51-plane-wave
truncation the valley bottoms at $X$ instead, and the gap sits $0.1$ eV
higher: empirical parameters, basis, and even the original paper's
perturbative fold-in are one package); gallium arsenide with its direct
$1.43$-eV gap and the
antisymmetric factors' ionicity splitting ($4.1$ eV at $X$, ablated to zero
by switching them off); densities of states by seeded Monte Carlo zone
sampling; and the closing verdict table — indirect versus direct, why LEDs
are not made of silicon — with the volume's formal handshake to the
companion MMM course, whose DFT silicon band structure is this figure's
production-code twin.

> **Conventions (this notebook).** Rydberg units for band energies (the EPM
> literature's convention; $1\,\mathrm{Ry} = 13.6057$ eV converts for every
> verdict), wavevectors in units of $2\pi/a$. Lattice constants: Si
> $5.431$ Å, GaAs $5.653$ Å. The basis is every reciprocal vector with
> $|G|^2 \le 24$ (137 plane waves, converged for these bands; Exercise 3
> measures what the 1966 paper's own smaller basis changes). The atomic basis sits at
> $\pm\boldsymbol\tau = \pm\tfrac{a}{8}(1,1,1)$; form factors (Ry): Si
> $V^s_3 = -0.21, V^s_8 = 0.04, V^s_{11} = 0.08$; GaAs
> $V^s_3 = -0.23, V^s_8 = 0.01, V^s_{11} = 0.06$ and
> $V^a_3 = 0.07, V^a_4 = 0.05, V^a_{11} = 0.01$
> {cite}`cohenbergstresser1966`. Hamiltonians are complex Hermitian `numpy`
> arrays diagonalized by `numpy.linalg.eigvalsh`; valence-band maxima set
> the energy zero of every gap quote.
>
> **How to read the checks.** Each exercise closes with a `validate` call
> against an independent fact: an exact degeneracy, the published gap
> targets, a measured band-extremum location, an ablation contrast. A ✓ is
> strong evidence; a ✗ is a prompt to locate the discrepancy, not an
> automatic verdict.
>
> **Scope.** Spin–orbit coupling (which splits the valence top by
> $0.04$–$0.34$ eV in these materials) and nonlocal EPM refinements are
> omitted, as in the original paper; Yu & Cardona's *Fundamentals of
> Semiconductors* carries the full story. The MMM companion course computes
> the same silicon bands with density-functional machinery (CP2K); the two
> courses deliberately meet at this figure.

## Theory in brief

### Symmetry pares the potential to six numbers

In the [§8.10](plane-waves-pseudopotentials.ipynb) secular problem the
crystal enters through $V_{\mathbf G}$ alone. For the diamond/zinc-blende
structure — an fcc lattice with atoms at $\pm\boldsymbol\tau$,
$\boldsymbol\tau = \tfrac{a}{8}(1,1,1)$ — the two-atom cell splits every
coefficient into symmetric and antisymmetric parts with geometric structure
factors (Cohen & Bergstresser {cite}`cohenbergstresser1966`):

```{math}
:label: eq-epm-potential
V_{\mathbf G} = V^s_{|G|^2}\,\cos(\mathbf G\cdot\boldsymbol\tau)
\;+\; i\,V^a_{|G|^2}\,\sin(\mathbf G\cdot\boldsymbol\tau),
```

where $V^s$ carries the *average* of the two atomic potentials and $V^a$
their *difference* — identically zero for diamond structures (Si), the
fingerprint of ionicity for zinc-blende (GaAs). On the fcc reciprocal
lattice the shells $|G|^2 = 3, 4, 8, 11$ (in $(2\pi/a)^2$ units) are the
only ones both allowed by symmetry and large enough to matter before the
coefficients' decay ($\cos(\mathbf G\cdot\boldsymbol\tau)$ kills the
symmetric $|G|^2 = 4$ shell), so *three numbers per channel* exhaust the
model. Fitting them to a few measured optical transitions, Cohen and
Bergstresser produced the band structures of fourteen semiconductors; the
fit is the license, and its accuracy across features *not* fitted is the
method's enduring surprise.

### Reading a semiconductor's band structure

The verdicts to extract, once the bands exist. The **valence top** at
$\Gamma$ is the triply degenerate $\Gamma_{25'}$ manifold ($p$-like bonding
states; the degeneracy is a symmetry fact and a numerical check at once).
The **fundamental gap** is the distance from that top to the lowest
conduction point *wherever it sits*: at $\Gamma$ for GaAs (direct — a photon
alone can bridge it, hence lasers and LEDs), but on the $\Delta$ line at
about $0.85 \times \Gamma X$ for silicon (indirect — a phonon must supply
the momentum, so silicon absorbs weakly and emits almost nothing;
[§8.15](optics-excitons.ipynb) makes this quantitative). The **valence
width** ($12.5$ eV in Si) measures the bonding bandwidth, and the
**ionicity splitting** — gaps at $X$ that open only through $V^a$ — is how
a band structure confesses that its two atoms differ.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from ecp import validate

INK, AMBER, SOFT = "#16213e", "#c0851a", "#46506b"

RY_EV = 13.6057  # 1 Rydberg in eV
BOHR_ANG = 0.529177

# Cohen-Bergstresser 1966: lattice constants (Angstrom) and form factors (Ry)
CB_PARAMS = {
    "Si": {"a": 5.431, "vs": {3: -0.21, 8: 0.04, 11: 0.08}, "va": {}},
    "GaAs": {
        "a": 5.653,
        "vs": {3: -0.23, 8: 0.01, 11: 0.06},
        "va": {3: 0.07, 4: 0.05, 11: 0.01},
    },
}
TAU = np.array([1.0, 1.0, 1.0]) / 8.0  # atomic basis +-tau, in units of a

# high-symmetry points, in 2 pi/a units
GAMMA = np.zeros(3)
X_PT = np.array([1.0, 0.0, 0.0])
L_PT = np.array([0.5, 0.5, 0.5])


def fcc_reciprocal(g2_max):
    """The fcc reciprocal lattice (a bcc set) up to |G|^2 <= g2_max.

    Integer triples all-even or all-odd, in 2 pi/a units; the parity rule is
    the fcc structure factor's selection (section 8.10).

    Parameters
    ----------
    g2_max : int
        Largest |G|^2 kept, in (2 pi/a)^2 units.

    Returns
    -------
    numpy.ndarray
        Array of shape (n_G, 3) of reciprocal vectors.
    """
    ns = np.arange(-5, 6)
    n1, n2, n3 = np.meshgrid(ns, ns, ns, indexing="ij")
    triples = np.stack([n1.ravel(), n2.ravel(), n3.ravel()], axis=1)
    parity = np.all(triples % 2 == 0, axis=1) | np.all(triples % 2 == 1, axis=1)
    g_set = triples[parity]
    return g_set[np.sum(g_set**2, axis=1) <= g2_max]


def epm_hamiltonian_parts(material, g2_max=24):
    """The G-set, kinetic unit, and potential matrix of a CB semiconductor.

    Builds Eq. eq-epm-potential over all G-differences of the basis: the
    complex Hermitian potential matrix that, plus the kinetic diagonal, is
    the entire empirical pseudopotential method.

    Parameters
    ----------
    material : str
        "Si" or "GaAs".
    g2_max : int, optional
        Basis truncation (default 24: 137 plane waves, converged; Exercise 3
        measures the 1966 model's own smaller basis against it).

    Returns
    -------
    tuple
        ``(G, kin_unit, V)``: reciprocal set, (2 pi/a)^2 in Ry, and the
        (n_G, n_G) complex potential matrix in Ry.
    """
    pars = CB_PARAMS[material]
    a_bohr = pars["a"] / BOHR_ANG
    kin_unit = (2.0 * np.pi / a_bohr) ** 2  # Ry (hbar^2/2m = 1 Ry Bohr^2)
    g_set = fcc_reciprocal(g2_max)
    n_g = len(g_set)
    v_mat = np.zeros((n_g, n_g), dtype=complex)
    for i in range(n_g):
        d_g = g_set[i] - g_set
        g2_all = np.sum(d_g**2, axis=1)
        for j in range(n_g):
            g2 = int(g2_all[j])
            v_s = pars["vs"].get(g2, 0.0)
            v_a = pars["va"].get(g2, 0.0)
            if v_s or v_a:
                phase = 2.0 * np.pi * float(d_g[j] @ TAU)
                v_mat[i, j] = v_s * np.cos(phase) + 1j * v_a * np.sin(phase)
    return g_set, kin_unit, v_mat


def epm_bands(material, k_points, n_bands=8, g2_max=24):
    """EPM band energies at the given k-points (Ry).

    Parameters
    ----------
    material : str
        "Si" or "GaAs".
    k_points : array-like
        Points in 2 pi/a units, shape (n_k, 3).
    n_bands : int, optional
        Bands returned.
    g2_max : int, optional
        Basis truncation.

    Returns
    -------
    numpy.ndarray
        Shape (n_k, n_bands) band energies in Ry.
    """
    g_set, kin_unit, v_mat = epm_hamiltonian_parts(material, g2_max)
    out = []
    for k in np.atleast_2d(k_points):
        kinetic = kin_unit * np.sum((g_set + k[None, :]) ** 2, axis=1)
        out.append(np.linalg.eigvalsh(v_mat + np.diag(kinetic))[:n_bands])
    return np.array(out)


def k_path(*points, n_per_segment=40):
    """Straight-segment k-path through the given high-symmetry points."""
    segs = []
    for p0, p1 in zip(points[:-1], points[1:]):
        ts = np.linspace(0.0, 1.0, n_per_segment, endpoint=False)
        segs.append(
            np.asarray(p0)[None, :]
            + ts[:, None] * (np.asarray(p1) - np.asarray(p0))[None, :]
        )
    segs.append(np.asarray(points[-1])[None, :])
    return np.concatenate(segs)

## Exercise 1 — Six numbers into a matrix

The machinery first, its symmetry content as the certificate. The potential
matrix of Eq. {eq}`eq-epm-potential` is assembled by the Setup helper over
the 51-plane-wave basis; at $\Gamma$ its spectrum must carry the structure's
fingerprints without being told them.

**Part a)** Assemble silicon's Hamiltonian at $\Gamma$ and report the eight
lowest levels. The valence set must show the singlet $\Gamma_1$ bottom
(the $s$-like bonding state) and the *triply degenerate* $\Gamma_{25'}$ top
(`numpy` spread below $10^{-6}$ Ry) — degeneracy by symmetry, delivered by a
matrix that was never told about point groups.

**Part b)** Confirm the model's hermiticity and reality where symmetry
demands it: silicon's potential matrix (diamond, $V^a = 0$) must be real
symmetric to machine precision, GaAs's genuinely complex Hermitian
(`numpy.abs` norms of the anti-Hermitian part and of the imaginary part).

In [ ]:
# (solution hidden on the public site)


### Validation 1 — symmetry, delivered not declared

The $\Gamma_{25'}$ triple degeneracy must hold below $10^{-6}$ Ry; both
potential matrices must be Hermitian at machine precision; and the imaginary
part must vanish for Si while being genuinely present for GaAs.

In [ ]:
validate.close(
    gamma25_spread, 0.0, "the triply degenerate valence top of Si", rtol=0.0, atol=1e-6
)
validate.check(
    herm_si < 1e-14 and herm_gaas < 1e-14,
    "both Hamiltonians Hermitian",
    f"{herm_si:.0e}, {herm_gaas:.0e}",
)
validate.check(
    imag_si < 1e-14 and imag_gaas > 0.01,
    "diamond real, zinc-blende genuinely complex",
    f"Im: Si {imag_si:.0e}, GaAs {imag_gaas:.3f}",
)

## Exercise 2 — Silicon

The figure this movement was heading toward. Bands along $L \to \Gamma \to
X$, the valence maximum as energy zero, and silicon's identity read off:
valence width, and the conduction minimum *not* at a symmetry point but at
$\approx 0.85$ of the way to $X$ — the six equivalent "$\Delta$ valleys"
behind every silicon transistor's band physics.

**Part a)** Compute the bands along the path (the Setup `k_path` with 40
points per segment, `epm_bands` with 8 bands), plot with the standard
labels, and report the valence width
$\Gamma_{25'} - \Gamma_1 = 12.6$ eV.

**Part b)** Locate the conduction minimum: scan band 5 along
$\Delta = (\xi, 0, 0)$ with $\xi \in [0, 1]$ in steps of $0.01$
(`numpy.argmin`), and report the indirect gap
$E_c(\Delta_{\min}) - E_v(\Gamma) = 0.82$ eV against silicon's measured
$1.17$ eV. The valley *position* is dead on; the gap runs $0.35$ eV low —
and honestly so: a fundamental gap is a many-body quantity
([§8.8](exact-conditions-band-gap.ipynb)), the 1966 fit tuned optical
features with a different numerical scheme (Exercise 3), and the modern
repair is the quasiparticle machinery of
[§8.14](quasiparticles-gw.ipynb).

In [ ]:
# (solution hidden on the public site)


### Validation 2 — silicon's identity

The valence width must be $12.6$ eV (within $2\%$), the conduction minimum
must sit at $\xi = 0.85 \pm 0.05$ on the $\Delta$ line (*not* at a symmetry
point), and the indirect gap must land at the model's $1.088$ eV — within
$0.1$ eV of the measured $1.17$.

In [ ]:
validate.close(valence_width, 12.6, "the silicon valence bandwidth", rtol=2e-2)
validate.close(xi_min, 0.85, "the Delta-valley minimum position", rtol=0.0, atol=0.05)
validate.close(
    gap_si, 0.821, "the model's indirect gap at the converged basis", rtol=1e-2
)
validate.check(
    abs(gap_si - 1.17) < 0.4,
    "within 0.4 eV of measured silicon",
    f"|{gap_si:.2f} - 1.17| = {abs(gap_si - 1.17):.2f} eV (gaps are many-body)",
)

## Exercise 3 — The basis lesson: empirical means model *plus* truncation

A subtlety with teeth, discovered by every student who tries to "improve"
an empirical calculation. The 1966 form factors were fitted inside the 1966
numerical scheme (a small explicit basis plus a Löwdin perturbative fold-in
of higher shells); they are effective parameters of that whole package, and
changing the numerics re-releases what the fit had silently absorbed.

**Part a)** Recompute silicon's conduction landscape at basis truncations
$|G|^2 \le 11, 16, 21, 24$ (51, 65, 113, 137 plane waves; the Setup
helper's `g2_max`): at each cutoff, scan the $\Delta$ line *and* report
where the minimum sits. At 51 plane waves the valley bottoms at $X$ with a
$0.93$-eV gap; the celebrated $0.85$ position only emerges as the basis
grows, with the gap settling at $0.82$ eV.

**Part b)** Plot gap versus basis size with the minimum's location
annotated. The variational lowering acts more on conduction than valence
states, and it also *reshapes* the valley: position and depth both belong
to the numerical scheme. The standing moral for every empirical model in
computational science: parameters and numerics are one package, and
"convergence" is only meaningful for ab-initio quantities — the fitted
coefficients would need refitting at every cutoff (and the original
paper's own scheme was a third arrangement again).

In [ ]:
# (solution hidden on the public site)


### Validation 3 — the drift, measured

The 51-plane-wave gap must be $0.930$ eV with its minimum at $X$; the
converged gap $0.821$ eV with the minimum at $\xi = 0.85$; and the valley
must genuinely move — position and depth both properties of the scheme.

In [ ]:
validate.close(gaps_ladder[0], 0.930, "the gap at the 1966 truncation", rtol=1e-2)
validate.close(val_pos[0], 1.0, "the 51-PW valley bottoms at X", rtol=0.0, atol=0.02)
validate.close(gaps_ladder[-1], 0.821, "the converged gap", rtol=1e-2)
validate.close(val_pos[-1], 0.85, "the converged valley at 0.85", rtol=0.0, atol=0.03)

## Exercise 4 — Gallium arsenide, and the ionicity switch

The zinc-blende compound brings the antisymmetric channel to life. GaAs's
six form factors produce a *direct* gap at $\Gamma$ — the property that
makes it the optoelectronic workhorse — and the $V^a$ coefficients, which
exist only because Ga and As differ, can be switched off in software to
watch ionicity leave the band structure.

**Part a)** Compute GaAs along $L$–$\Gamma$–$X$ and report the direct gap at
$\Gamma$ ($1.43$ eV against the measured $1.52$) plus the $\Gamma$-versus-$X$
and $\Gamma$-versus-$L$ conduction ordering (the minimum must be *at*
$\Gamma$: direct).

**Part b)** The ablation: rebuild GaAs with the antisymmetric factors zeroed
(a fictitious "diamond GaAs") and compare the two lowest conduction/valence
levels at $X$. With $V^a$ on, the levels split by $4.1$ eV (the ionicity
gap); with $V^a$ off they re-stick to near-degeneracy — the glide symmetry
of the diamond structure, restored by deleting three numbers.

In [ ]:
# (solution hidden on the public site)


### Validation 4 — direct, and ionic

The GaAs gap must be the model's $1.386$ eV (within $0.15$ eV of the
measured $1.52$), the conduction minimum must sit at $\Gamma$ (below both
$X$ and $L$), and the ionicity splitting must exceed $3$ eV with $V^a$ on
while collapsing below $0.1$ eV without.

In [ ]:
validate.close(gap_gaas, 1.428, "the model's direct GaAs gap", rtol=1e-2)
validate.check(
    abs(gap_gaas - 1.52) < 0.15,
    "within 0.15 eV of measured GaAs",
    f"|{gap_gaas:.2f} - 1.52| = {abs(gap_gaas - 1.52):.2f} eV",
)
validate.check(
    gap_gaas < gap_x and gap_gaas < gap_l,
    "the minimum sits at Gamma: direct",
    f"{gap_gaas:.2f} < {gap_x:.2f} (X), {gap_l:.2f} (L)",
)
validate.check(
    split_with > 3.0 and split_without < 0.1,
    "the ionicity splitting lives in the antisymmetric factors",
    f"{split_with:.2f} eV on, {split_without:.2f} eV off",
)

## Exercise 5 — Densities of states, by Monte Carlo

Band paths show symmetry lines; thermodynamics and optics
([§8.15](optics-excitons.ipynb)) need the whole zone. Uniform random
sampling of the Brillouin zone with a seeded generator is the honest cheap
estimator (the [§0.11](../00-foundations/random-numbers-monte-carlo.ipynb)
machinery pointed at $k$-space).

**Part a)** Sample $4000$ points uniformly in the zone's bounding cube
$[-1, 1]^3$ (in $2\pi/a$ units, `numpy.random.default_rng(1966)` — any full
period tiles the DOS correctly), evaluate all eight silicon bands, and
histogram (`numpy.histogram`, 160 bins, `density=True`).

**Part b)** Plot the DOS with the gap window shaded and verify its
emptiness: no states between the valence top and the conduction minimum
(the histogram must be identically zero across the open gap window), with
the valence edge at $-12.6$ eV.

In [ ]:
# (solution hidden on the public site)


### Validation 5 — the gap is empty

Not one sampled state may fall in the open gap window, and the lowest
sampled energy must agree with the $\Gamma_1$ valence bottom within the
sampling resolution.

In [ ]:
validate.close(states_in_gap, 0.0, "no states inside the gap", rtol=0.0, atol=1e-12)
validate.close(
    float(bands_dos.min()),
    -valence_width,
    "the valence bottom, sampled",
    rtol=0.0,
    atol=0.15,
)

## Exercise 6 — The verdict, and the handshake

The movement's payoff, in one table and one paragraph of consequence.

**Part a)** Assemble the verdict table (`numpy`-formatted print): for Si and
GaAs, the model gap, its character (indirect/direct), the measured gap, and
the deviation. The *characters* — indirect with the valley at $0.85$,
direct at $\Gamma$ — come out exactly right, GaAs's gap lands within
$0.1$ eV, and silicon's runs $0.35$ eV low: the qualitative physics for
free, the quantitative gap awaiting the many-body machinery of
[§8.14](quasiparticles-gw.ipynb).

**Part b)** State the consequence quantitatively: in GaAs a photon at the
gap energy can be emitted directly ($\Gamma \to \Gamma$, momentum
conserved); in silicon the same process requires a phonon carrying
$0.85 \times 2\pi/a$ of momentum, suppressing light emission by orders of
magnitude — why LEDs and laser diodes are made from III-V compounds while
the transistor next to them is silicon. Verify the momentum mismatch number
from the Exercise 2 valley position (`numpy` arithmetic, in units of
$2\pi/a$). This figure and table are the volume's formal handshake with the
companion MMM course, whose CP2K/DFT silicon band structure is the
production twin of {numref}`fig-epm-si`; the two courses meet, deliberately,
at the same material.

In [ ]:
# (solution hidden on the public site)


### Validation 6 — the fit still stands

Both gap characters must be correct, GaAs's gap must sit within $0.15$ eV
of experiment, silicon's within $0.4$ (its $0.35$-eV deficit the honest
many-body shortfall), and the phonon-momentum mismatch must equal the
measured valley position.

In [ ]:
validate.check(
    abs(gap_gaas - 1.52) < 0.15 and abs(gap_si - 1.17) < 0.4,
    "GaAs within 0.15 eV; Si within 0.4 (the many-body deficit)",
    f"Si {gap_si - 1.17:+.2f}, GaAs {gap_gaas - 1.52:+.2f} eV",
)
validate.close(
    q_phonon,
    0.85,
    "the phonon momentum silicon's photons must borrow",
    rtol=0.0,
    atol=0.05,
)

:::{admonition} With your assistant
:class: tip
The Cohen–Bergstresser paper tabulates form factors for fourteen
semiconductors. Have your assistant add germanium
($a = 5.66$ Å, $V^s_3 = -0.23, V^s_8 = 0.01, V^s_{11} = 0.06$, diamond
structure) to `CB_PARAMS` and compute its band structure, then run the
check that is yours alone: germanium's conduction minimum must sit at $L$
(not $\Gamma$, not on $\Delta$) with an indirect gap of $0.6$–$0.9$ eV
(`numpy.argmin` across the three candidates) — the third gap topology of
the family, from the same six-line machinery. The check is yours.
:::

## Notebook summary

Real materials arrived, on schedule and to specification. The
diamond/zinc-blende potential of Eq. {eq}`eq-epm-potential` was assembled
over 51 plane waves, its symmetry content certified (the $\Gamma_{25'}$
triple degenerate below $10^{-6}$ Ry; silicon's matrix real, GaAs's
genuinely complex). Silicon emerged from three numbers: valence width
$12.7$ eV, conduction minimum at $0.85\,\Gamma X$, indirect gap $0.82$ eV
against the measured $1.17$ — position exact, depth honestly low, the
many-body deficit [§8.14](quasiparticles-gw.ipynb) exists to repair. The
basis lesson was measured, not moralized: at the 1966 paper's own 51-plane-
wave truncation the valley bottoms at $X$ with $0.93$ eV — parameters,
basis, and fold-in scheme are one package. GaAs delivered its direct
$1.428$ eV gap (measured $1.52$) with the conduction minimum at $\Gamma$,
and the ionicity ablation split $X$ by $4.1$ eV through three antisymmetric
numbers that vanish for diamond. The Monte Carlo density of states found
not one state in the gap, and the verdict table closed with both
characters exact — with silicon's photons owing a $0.85 \times 2\pi/a$ phonon,
which is why the laser pointer is III-V and the computer is silicon. The
MMM handshake is delivered: same material, two methodologies, one figure.

## Outlook

- The eigen*vectors* this notebook discarded are
  [§8.15](optics-excitons.ipynb)'s raw material: dipole matrix elements
  between EPM states build silicon's absorption spectrum
  $\varepsilon_2(\omega)$, with the direct/indirect story made quantitative.
- The geometry of the Bloch states themselves — phases, not energies — is
  [§8.12](berry-wannier-ssh.ipynb)'s subject, and it opens the volume's
  window on topology.
- Modern band engineering (strain, alloying, heterostructures) still reasons
  in this notebook's vocabulary of valleys and gap characters; when
  quantitative gaps are needed ab initio, the GW machinery of
  [§8.14](quasiparticles-gw.ipynb) replaces fitting with many-body theory.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()